# 05 — 학습 없이 갱신(F6)을 첫 시도 이득 $G$로 평가

[04_gain_metric](04_gain_metric.ipynb)의 계산을 그대로 쓰고, 비교군에 F6을 더합니다. F6은 운영 1.0의 가중치를 고정한 채 순서형 문턱 파라미터 4개만 window마다 다시 맞춥니다([training_free.py](training_free.py)). F6의 단계 선택은 저장된 logit에서 결정적으로 다시 계산하므로 GPU가 필요 없습니다. **Colab CPU 런타임**에서 실행합니다.

| 비교군 | 내용 |
|---|---|
| F0 ~ F5 | 04와 같음 |
| F6 | 학습 없이 갱신. 상향 없이 끝난 요청과 상향 기록을 모두 씀(고정 로거) |
| F6-on | F6과 같지만 F6이 직접 고른 단계의 결과로 갱신 |
| F6-esc | F6과 같은 모델에 상향 기록만 씀(현재 `/escalate` 계약으로 받을 수 있는 정보) |

bootstrap 난수 seed와 평가 데이터 순서가 04와 같아 F0~F5의 값과 구간이 04 결과와 같게 나와야 합니다. 마지막 셀에서 확인합니다. F6은 구간을 따라가며 갱신하므로 F1~F3과 같은 다음 구간 문항(prequential)이 주 평가이고, 구간 6까지 맞춘 파라미터를 최종 평가 세트(test, 7,202문항)에도 적용해 F4·F5와 같은 문항으로 비교합니다.


In [ ]:
# 환경 설정. Colab이면 Drive를 붙입니다. 다른 곳에서 돌릴 때는 LLM_ROUTER_DRIVE에 Drive의 LLM_Router 폴더 경로를 넣습니다.
import hashlib, json, os, sys, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

try:
    display
except NameError:  # 노트북 밖에서 실행할 때
    display = print

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/LLM_Router')
else:
    DRIVE = Path(os.environ['LLM_ROUTER_DRIVE'])

RUNS = {
    'run001': (DRIVE / 'retraining-effect/runs/retraining-effect-v1-run001', ['f1_cycle_only', 'f2_replay']),
    'f3': (DRIVE / 'retraining-effect-f3/runs/retraining-effect-v2-f3-run001', ['f3_balanced_easy']),
    'f4': (DRIVE / 'retraining-effect-f4/runs/retraining-effect-v3-f4-run001', ['f4_bidirectional']),
    'f5': (DRIVE / 'retraining-effect-f5/runs/retraining-effect-v4-f5-run001', ['f5_matched_prior']),
}
OUT_DIR = Path(os.environ.get('F6_GAIN_OUT', DRIVE / 'retraining-gain-f6-20261021'))
REFERENCE_DIR = DRIVE / 'retraining-gain-20261021'

FROZEN = 'f0_frozen'
F6_ARMS = ['f6_training_free', 'f6_training_free_on_policy', 'f6_escalation_only']
ARMS = [FROZEN, 'f1_cycle_only', 'f2_replay', 'f3_balanced_easy', 'f4_bidirectional', 'f5_matched_prior',
        *F6_ARMS]
SHORT = dict(zip(ARMS, ['F0', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F6-on', 'F6-esc']))
NAME = {FROZEN: '재학습 안 함', 'f1_cycle_only': '새 피드백만', 'f2_replay': '과거 데이터 섞기',
        'f3_balanced_easy': 'easy 보충', 'f4_bidirectional': '양방향 교정(오답만)',
        'f5_matched_prior': '양방향 교정 + 비율 맞춤', 'f6_training_free': '학습 없이 갱신',
        'f6_training_free_on_policy': '학습 없이 갱신(자기 선택으로 갱신)',
        'f6_escalation_only': '학습 없이 갱신(상향 기록만)'}
SEEDS = (42, 43, 44)
WINDOWS = range(2, 7)
SETS = ['prequential', 'gate', 'forget', 'test']   # 04와 같은 순서(bootstrap 난수 seed가 순서에 묶여 있음)

# F6 helper: training_free.py와, 그것이 쓰는 experiment.py(F3 kit에 든 판)를 불러온다.
TF_PIN = '7edd05c67d74d9b45bb43a4d8affbf59fa276aefb1c3ecf66c0c507c29de1cb2'
helper_dir = Path('/content/f6-gain') if IN_COLAB else Path(os.environ.get('F6_HELPER_DIR', '/tmp/f6-gain'))
helper_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE / 'retraining-effect-f3/source/retraining-source-f3.zip') as z:
    (helper_dir / 'experiment.py').write_bytes(z.read('ml/notebooks/retraining_effect/experiment.py'))
source = (DRIVE / 'retraining-effect-f6/training_free.py').read_bytes()
assert hashlib.sha256(source).hexdigest() == TF_PIN, 'training_free.py가 고정한 판과 다릅니다'
(helper_dir / 'training_free.py').write_bytes(source)
sys.path.insert(0, str(helper_dir))
import training_free as tf

for key, (root, _) in RUNS.items():
    print(f'{key:7s}', root, '있음' if root.is_dir() else '없음')
missing_runs = [k for k, (root, _) in RUNS.items() if not root.is_dir()]
if missing_runs:
    raise FileNotFoundError(f'run 폴더가 없습니다: {missing_runs}. DRIVE 경로를 확인하세요.')

## 입력 읽기와 검증

- 모든 예측 파일은 같은 폴더의 `receipt.json`에 적힌 SHA-256과 대조합니다. 다르면 멈춥니다.
- 같은 평가 데이터에서는 모든 비교군·seed의 문항 목록과 `success_*` 값이 같아야 합니다. 다르면 멈춥니다.
- F0의 window 예측은 run마다 `stream-windows.parquet`에 있습니다. 네 run의 값이 같은지도 확인합니다.

In [ ]:
TIERS = ('low', 'medium', 'high')
LABELS = ('easy', 'medium', 'hard')
S_COLS = [f'success_{t}' for t in TIERS]
KEEP = ['sample_id', 'source_dataset', 'duplicate_group_id', *S_COLS]


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


INPUT_SHA = {}


def read_pinned(directory):
    # receipt.json의 SHA와 맞는 예측 파일만 읽는다.
    path = directory / 'predictions.parquet'
    receipt = json.loads((directory / 'receipt.json').read_text())
    got = sha256(path)
    if got != receipt['predictions_sha256']:
        raise RuntimeError(f'SHA-256 불일치: {path}\n  receipt {receipt["predictions_sha256"]}\n  실제 {got}')
    INPUT_SHA[str(path.relative_to(DRIVE))] = got
    frame = pd.read_parquet(path, columns=KEEP + ['prediction'])
    return frame.sort_values('sample_id', kind='stable').reset_index(drop=True)


def as_tier(labels):
    tier = pd.Series(labels).map({name: i for i, name in enumerate(LABELS)})
    if tier.isna().any():
        raise ValueError('예측 라벨은 easy/medium/hard여야 합니다')
    return tier.to_numpy(int)


# 평가 데이터별: base(문항·성공 기록)와 {arm: {seed: 첫 시도 단계}}
data = {name: {'base': None, 'choice': {}} for name in SETS}


def add(set_name, arm, seed, frame, column='prediction'):
    entry = data[set_name]
    frame = frame.sort_values('sample_id', kind='stable').reset_index(drop=True)
    if entry['base'] is None:
        entry['base'] = frame[KEEP].copy()
    base = entry['base']
    if not frame['sample_id'].equals(base['sample_id']):
        raise RuntimeError(f'{set_name}/{arm}/{seed}: 문항 목록이 다릅니다')
    if not np.array_equal(frame[S_COLS].to_numpy(float), base[S_COLS].to_numpy(float)):
        raise RuntimeError(f'{set_name}/{arm}/{seed}: success 값이 다릅니다')
    seeds = entry['choice'].setdefault(arm, {})
    if seed in seeds:
        if not np.array_equal(seeds[seed], as_tier(frame[column])):
            raise RuntimeError(f'{set_name}/{arm}/{seed}: 같은 모델의 예측이 run마다 다릅니다')
        return
    seeds[seed] = as_tier(frame[column])


# F0 window 예측: 네 run의 stream이 같아야 한다.
streams = {}
for key, (root, _) in RUNS.items():
    path = root / 'shared-data/stream-windows.parquet'
    INPUT_SHA[str(path.relative_to(DRIVE))] = sha256(path)
    stream = pd.read_parquet(path, columns=KEEP + ['window', 'initial_label'])
    streams[key] = stream[stream.window.isin(WINDOWS)].sort_values('sample_id', kind='stable').reset_index(drop=True)
reference = streams['run001']
for key, stream in streams.items():
    if not stream[KEEP + ['window', 'initial_label']].equals(reference[KEEP + ['window', 'initial_label']]):
        raise RuntimeError(f'{key}의 stream이 run001과 다릅니다')
add('prequential', FROZEN, 0, reference, column='initial_label')
WINDOW_OF = reference.set_index('sample_id')['window']

# 재학습 비교군의 window 예측
for key, (root, arms) in RUNS.items():
    for arm in arms:
        for seed in SEEDS:
            parts = []
            for window in WINDOWS:
                directory = root / 'scores' / arm / f'seed-{seed}' / f'window-{window:02d}'
                parts.append(read_pinned(directory))
            add('prequential', arm, seed, pd.concat(parts, ignore_index=True))

# 최종 모델 예측: Drive에 있는 final/<역할>을 모두 찾는다(RouterArena 제외).
for key, (root, arms) in RUNS.items():
    for arm, seeds in [(FROZEN, [0])] + [(a, SEEDS) for a in arms]:
        for seed in seeds:
            final = root / 'scores' / arm / f'seed-{seed}' / 'final'
            if not final.is_dir():
                continue
            for role_dir in sorted(final.iterdir()):
                if role_dir.name in SETS and (role_dir / 'receipt.json').exists():
                    add(role_dir.name, arm, seed, read_pinned(role_dir))

summary_rows = []
for name in SETS:
    entry = data[name]
    if entry['base'] is None:
        continue
    for arm in ARMS:
        if arm in entry['choice']:
            summary_rows.append({'평가 데이터': name, '문항 수': len(entry['base']), '비교군': SHORT[arm],
                                 'seed': ', '.join(map(str, sorted(entry['choice'][arm])))})
display(pd.DataFrame(summary_rows))
print('검증한 예측 파일:', len(INPUT_SHA))

## F6 단계 선택 다시 계산

run001의 봉인된 stream(logit 포함)으로 F6 세 가지를 다시 돌립니다. window $j+1$은 window 1~$j$의 관측으로 맞춘 파라미터로 고릅니다. 최종 모델 평가(gate·forget)는 window 6까지 맞춘 파라미터를 F0의 최종 logit에 적용합니다. 결정적 계산이라 seed는 0 하나입니다.

In [ ]:
run_dir = RUNS['run001'][0]
plan = json.loads((run_dir / 'plan.json').read_text())
full_stream = pd.read_parquet(run_dir / 'shared-data/stream-windows.parquet')
frozen = full_stream[tf.LOGIT_COLUMNS].to_numpy(float).argmax(1)
assert np.array_equal(frozen, as_tier(full_stream['initial_label'])), 'logit이 봉인된 로거 예측을 재현하지 않습니다'
modes = {'f6_training_free': 'fixed_logger', 'f6_training_free_on_policy': 'on_policy',
         'f6_escalation_only': 'escalation_only'}
f6_params, f6_fits = {}, {}
for arm, mode in modes.items():
    decisions, params, fits = tf.run_stream(full_stream, plan['config']['windows'], mode)
    f6_params[arm], f6_fits[arm] = params, fits
    later = full_stream.window.isin(WINDOWS).to_numpy()
    frame = full_stream.loc[later, KEEP].copy()
    frame['prediction'] = np.asarray(LABELS)[decisions[later]]
    add('prequential', arm, 0, frame)

for role in ('gate', 'forget', 'test'):
    # F0의 최종 예측(logit)이 있는 첫 run을 쓴다. test는 F4·F5 run에만 있다.
    directory = next(root / 'scores' / FROZEN / 'seed-0' / 'final' / role for root, _ in RUNS.values()
                     if (root / 'scores' / FROZEN / 'seed-0' / 'final' / role / 'receipt.json').exists())
    path = directory / 'predictions.parquet'
    receipt = json.loads((directory / 'receipt.json').read_text())
    assert sha256(path) == receipt['predictions_sha256'], path
    final = pd.read_parquet(path)
    for arm, params in f6_params.items():
        frame = final[KEEP].copy()
        frame['prediction'] = np.asarray(LABELS)[tf.decide(params, tf.features(final))]
        add(role, arm, 0, frame)
print({SHORT[a]: tf.params_record(p) for a, p in f6_params.items()})

## 계산 함수

`gain`은 R0~R3 노트북과 같은 계산입니다. bootstrap에서는 같은 계산을 행렬로 한 번에 합니다(`baseline_many`). 아래 셀 끝에서 두 계산이 같은 값을 내는지 확인합니다.

In [ ]:
PRICE = np.array([0.2, 0.6, 1.0])          # 1:3:5, high = 1
N_BOOT = 2000
SEED = 20261021


def baseline(cost, fixed_acc):
    # 비용 cost 이하에서 고정 단계 혼합이 낼 수 있는 최고 첫 시도 해결률(위로 볼록한 껍질, 최고점 뒤는 평평).
    a_low, a_med, a_high = fixed_acc
    chord = a_low + (a_high - a_low) * (PRICE[1] - PRICE[0]) / (PRICE[2] - PRICE[0])
    keep = [0, 1, 2] if a_med > chord else [0, 2]
    xs, ys = PRICE[keep], np.asarray(fixed_acc)[keep]
    top = int(np.argmax(ys))
    return float(np.interp(cost, xs[:top + 1], ys[:top + 1]))


def baseline_many(cost, fixed):
    # baseline의 행렬판. cost: (B,), fixed: (B, 3)
    lo, me, hi = fixed[:, 0], fixed[:, 1], fixed[:, 2]
    chord = lo + (hi - lo) * (PRICE[1] - PRICE[0]) / (PRICE[2] - PRICE[0])
    on_hull = me > chord
    top3 = PRICE[np.argmax(fixed, axis=1)]
    top2 = np.where(hi > lo, PRICE[2], PRICE[0])
    c = np.minimum(cost, np.where(on_hull, top3, top2))

    def seg(x0, y0, x1, y1):
        return y0 + (y1 - y0) * (np.clip(c, x0, x1) - x0) / (x1 - x0)

    three = np.where(c <= PRICE[1], seg(PRICE[0], lo, PRICE[1], me), seg(PRICE[1], me, PRICE[2], hi))
    return np.where(on_hull, three, seg(PRICE[0], lo, PRICE[2], hi))


def gain(choice, s):
    rows = np.arange(len(s))
    acc = s[rows, choice].mean()
    cost = PRICE[choice].mean()
    h = baseline(cost, s.mean(axis=0))
    rates = np.bincount(choice, minlength=3) / len(choice)
    return {'A': acc, 'C': cost, 'H': h, 'G': acc - h,
            'low_rate': rates[0], 'medium_rate': rates[1], 'high_rate': rates[2]}


def oracle_choice(s):
    # 맞히는 가장 싼 단계, 아무도 못 맞히면 low.
    first = np.argmax(s > 0, axis=1)
    return np.where(s.any(axis=1), first, 0)


def group_weights(groups, set_index, source_index):
    # 질문 묶음 단위 재표집: (N_BOOT, 문항 수) 가중치. 같은 (평가 데이터, 원천)이면 모든 비교군이 같은 재표집을 쓴다.
    codes, uniq = pd.factorize(groups, sort=True)
    rng = np.random.default_rng([SEED, set_index, source_index])
    draws = rng.integers(0, len(uniq), size=(N_BOOT, len(uniq)))
    counts = np.zeros((N_BOOT, len(uniq)), dtype=np.float32)
    np.add.at(counts, (np.arange(N_BOOT)[:, None], draws), 1.0)
    return counts[:, codes]


def boot_gain(choice, s, weights, fixed):
    total = weights.sum(axis=1)
    acc = weights @ s[np.arange(len(s)), choice].astype(np.float32) / total
    cost = weights @ PRICE[choice].astype(np.float32) / total
    return acc - baseline_many(cost, fixed)


# 두 계산이 같은지 확인: 무작위 성공 기록과 경계 사례(medium이 껍질 밖, high가 가장 낮음)
rng = np.random.default_rng(0)
for trial in range(300):
    fixed_acc = rng.random(3)
    if trial % 3 == 0:
        fixed_acc[1] = min(fixed_acc[0], fixed_acc[2]) * 0.5
    if trial % 5 == 0:
        fixed_acc[2] = fixed_acc.min() * 0.5
    cost = rng.uniform(0.2, 1.0)
    one = baseline(cost, fixed_acc)
    many = baseline_many(np.array([cost]), fixed_acc[None, :])[0]
    assert abs(one - many) < 1e-9, (fixed_acc, cost, one, many)
print('baseline 행렬판 확인: 300개 사례 일치')

## 실행

평가 데이터 × 원천마다 점 추정과 bootstrap을 계산하고, 원천 평균(source-macro)과 seed 평균을 냅니다. 몇 분 걸립니다.

In [ ]:
point_rows, window_rows, results = [], [], {}
for set_index, name in enumerate(SETS):
    entry = data[name]
    if entry['base'] is None:
        print(f'[건너뜀] {name}: 예측이 없습니다')
        continue
    base = entry['base']
    sources = sorted(base['source_dataset'].unique())
    boots = {}                                 # (arm, seed) -> (source 수, N_BOOT)
    for source_index, source in enumerate(sources):
        mask = base['source_dataset'].eq(source).to_numpy()
        s = base.loc[mask, S_COLS].to_numpy(float)
        weights = group_weights(base.loc[mask, 'duplicate_group_id'].to_numpy(), set_index, source_index)
        fixed = (weights @ s.astype(np.float32)) / weights.sum(axis=1, keepdims=True)
        for arm, seeds in entry['choice'].items():
            for seed, choice in seeds.items():
                ch = choice[mask]
                point_rows.append({'set': name, 'source': source, 'arm': arm, 'seed': seed, 'n': int(mask.sum()),
                                   **gain(ch, s)})
                boots.setdefault((arm, seed), []).append(boot_gain(ch, s, weights, fixed))
                if name == 'prequential':
                    windows = WINDOW_OF.loc[base.loc[mask, 'sample_id']].to_numpy()
                    for window in WINDOWS:
                        w = windows == window
                        window_rows.append({'source': source, 'arm': arm, 'seed': seed, 'window': window,
                                            **gain(ch[w], s[w])})
        point_rows.append({'set': name, 'source': source, 'arm': 'oracle', 'seed': 0, 'n': int(mask.sum()),
                           **gain(oracle_choice(s), s)})
    # 원천 평균 → seed 평균
    macro = {key: np.mean(v, axis=0) for key, v in boots.items()}
    results[name] = {arm: np.mean([macro[(arm, seed)] for seed in seeds], axis=0)
                     for arm, seeds in entry['choice'].items()}
    print(f'{name}: 문항 {len(base):,}, 원천 {sources}, 비교군 {[SHORT[a] for a in entry["choice"]]}')

points = pd.DataFrame(point_rows)
windows = pd.DataFrame(window_rows)

## 결과

### 비교군별 $G$ (원천 평균, seed 평균)

`G_lo`·`G_hi`는 95% 구간, `G_seed_min`·`G_seed_max`는 seed 3개 중 가장 낮은·높은 값입니다. `A`·`C`·`H`와 단계 비율도 원천 평균입니다.

In [ ]:
METRICS = ['A', 'C', 'H', 'G', 'low_rate', 'medium_rate', 'high_rate']
per_seed = (points.groupby(['set', 'arm', 'seed'])[METRICS].mean().reset_index())   # 원천 평균
summary = (per_seed.groupby(['set', 'arm'])
           .agg(**{m: (m, 'mean') for m in METRICS}, G_seed_min=('G', 'min'), G_seed_max=('G', 'max'),
                seeds=('seed', 'count'))
           .reset_index())
summary['G_lo'] = np.nan
summary['G_hi'] = np.nan
for i, row in summary.iterrows():
    if row.arm in results.get(row.set, {}):
        summary.loc[i, ['G_lo', 'G_hi']] = np.quantile(results[row.set][row.arm], [0.025, 0.975])
order = {a: i for i, a in enumerate(ARMS + ['oracle'])}
summary = summary.sort_values(['set', 'arm'], key=lambda c: c.map(order) if c.name == 'arm' else c.map(SETS.index))
summary.insert(2, '이름', summary.arm.map({**NAME, 'oracle': '참고: 맞히는 가장 싼 단계'}))
summary['arm'] = summary.arm.map({**SHORT, 'oracle': 'oracle'})
pd.options.display.float_format = '{:.3f}'.format
display(summary[['set', 'arm', '이름', 'G', 'G_lo', 'G_hi', 'G_seed_min', 'G_seed_max', 'A', 'C', 'H',
                 'low_rate', 'medium_rate', 'high_rate', 'seeds']])

### 기존 모델 대비 변화 $\Delta G = G_\text{재학습} - G_\text{F0}$

`separated = True`는 95% 구간이 0을 포함하지 않는다는 뜻입니다. 양수면 재학습이 F0보다 첫 시도에서 단계를 더 잘 골랐다는 뜻입니다.

In [ ]:
delta_rows = []
for name, arms in results.items():
    if FROZEN not in arms:
        continue
    f0 = arms[FROZEN]
    for arm, values in arms.items():
        if arm == FROZEN:
            continue
        diff = values - f0
        lo, hi = np.quantile(diff, [0.025, 0.975])
        point = summary.loc[(summary.set == name) & (summary.arm == SHORT[arm]), 'G'].item() \
            - summary.loc[(summary.set == name) & (summary.arm == 'F0'), 'G'].item()
        delta_rows.append({'set': name, 'arm': SHORT[arm], '이름': NAME[arm], 'delta_G': point,
                           'lo': lo, 'hi': hi, 'separated': bool(lo > 0 or hi < 0)})
deltas = pd.DataFrame(delta_rows)
display(deltas)

### window별 $G$ (prequential, 원천 평균, seed 평균)

재학습이 window를 거치며 나아지거나 나빠지는지 봅니다. 점 추정만 냅니다.

In [ ]:
by_window = (windows.groupby(['arm', 'seed', 'window'])['G'].mean()      # 원천 평균
             .groupby(['arm', 'window']).mean().unstack('window'))
by_window.index = by_window.index.map(SHORT)
display(by_window.loc[[SHORT[a] for a in ARMS if SHORT[a] in by_window.index]])

## 그림

왼쪽부터 평가 데이터별 $G$와 95% 구간입니다. 회색 점선은 0(같은 비용의 고정 단계 혼합과 같음)입니다. Colab 기본 글꼴로 한글이 깨지지 않도록 그림 안의 글자는 영문으로 씁니다.

In [ ]:
import matplotlib.pyplot as plt

shown = [name for name in SETS if name in results]
fig, axes = plt.subplots(1, len(shown), figsize=(4.2 * len(shown), 3.6), sharey=False)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, shown):
    part = summary[(summary.set == name) & (summary.arm != 'oracle')]
    y = np.arange(len(part))[::-1]
    colors = ['#4c43c4' if a.startswith('F6') else '#686d78' for a in part.arm]
    ax.hlines(y, part.G_lo, part.G_hi, color=colors, lw=1.8)
    ax.scatter(part.G, y, color=colors, zorder=3)
    for yy, g in zip(y, part.G):
        ax.annotate(f'{g:+.3f}', (g, yy), xytext=(0, 7), textcoords='offset points', ha='center', fontsize=8)
    ax.axvline(0, color='#b8bdc8', ls='--', lw=1)
    ax.set_yticks(y, part.arm)
    ax.set_title(f'{name} (n={len(data[name]["base"]):,})', fontsize=10)
    ax.set_xlabel('first-try gain G')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
fig.tight_layout()
plt.show()

## 저장

`OUT_DIR`(`MyDrive/LLM_Router/retraining-gain-20261021/`)에 표·그림·요약·manifest를 저장합니다. 같은 입력이면 같은 결과가 나오므로 다시 실행하면 덮어씁니다.

## 04 결과와 같은지 확인

F0~F5의 $G$는 04의 `gain-summary.csv`와 같아야 하고, 95% 구간은 float32 합산 순서 차이(10⁻⁵ 미만)만 허용합니다. 그래야 bootstrap 재표집이 공유된 것입니다.

In [ ]:
reference = pd.read_csv(REFERENCE_DIR / 'gain-summary.csv')
mine = summary[summary.arm.isin(['F0', 'F1', 'F2', 'F3', 'F4', 'F5'])][['set', 'arm', 'G', 'G_lo', 'G_hi']]
merged = mine.merge(reference[['set', 'arm', 'G', 'G_lo', 'G_hi']], on=['set', 'arm'], suffixes=('', '_04'))
assert len(merged) == len(mine), '04 결과에 없는 행이 있습니다'
# 점 추정은 같아야 하고, 구간은 float32 합산 순서 차이만 허용한다.
for col, tol in (('G', 1e-12), ('G_lo', 1e-5), ('G_hi', 1e-5)):
    gap = (merged[col] - merged[f'{col}_04']).abs().max()
    assert gap < tol, f'{col}이 04와 {gap} 다릅니다'
print('F0~F5가 04 결과와 일치:', len(merged), '행')

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT_DIR / 'gain-summary.csv', index=False)
deltas.to_csv(OUT_DIR / 'delta-vs-f0.csv', index=False)
points.to_csv(OUT_DIR / 'points-by-source-seed.csv', index=False)
windows.to_csv(OUT_DIR / 'windows-by-source-seed.csv', index=False)
fig.savefig(OUT_DIR / 'gain-by-set.png', dpi=200, bbox_inches='tight')


def table(frame, columns):
    head = '| ' + ' | '.join(columns) + ' |\n|' + '---|' * len(columns) + '\n'
    body = ''.join('| ' + ' | '.join(f'{v:+.3f}' if isinstance(v, float) else str(v) for v in row) + ' |\n'
                   for row in frame[columns].itertuples(index=False))
    return head + body


lines = ['# 재학습 실험 첫 시도 이득 G (자동 생성)\n',
         f'bootstrap {N_BOOT}회(질문 묶음 단위, seed {SEED}), 단계 가격 1:3:5, 원천 평균 → seed 평균.\n']
for name in shown:
    lines.append(f'\n## {name} (문항 {len(data[name]["base"]):,})\n\n')
    lines.append(table(summary[summary.set == name], ['arm', '이름', 'G', 'G_lo', 'G_hi', 'A', 'C']))
    part = deltas[deltas.set == name]
    if len(part):
        lines.append('\nF0 대비 ΔG\n\n' + table(part, ['arm', '이름', 'delta_G', 'lo', 'hi', 'separated']))
(OUT_DIR / 'gain-summary.md').write_text(''.join(lines), encoding='utf-8')

manifest = {
    'notebook': 'ml/notebooks/retraining_effect/05_f6_gain_metric.ipynb',
    'metric': 'G = A - H(C), first-try success, tier price 0.2/0.6/1.0, upper concave hull of fixed tiers',
    'aggregation': 'per source then equal-weight mean over sources, then mean over seeds',
    'bootstrap': {'draws': N_BOOT, 'seed': SEED, 'unit': 'duplicate_group_id within source',
                  'paired': 'shared resamples across arms and seeds'},
    'runs': {k: str(root.relative_to(DRIVE)) for k, (root, _) in RUNS.items()},
    'f6_helper_sha256': TF_PIN,
    'f6_params': {SHORT[a]: tf.params_record(p) for a, p in f6_params.items()},
    'sets': {name: len(data[name]['base']) for name in shown},
    'input_sha256': INPUT_SHA,
    'output_sha256': {p.name: sha256(p) for p in sorted(OUT_DIR.iterdir())
                      if p.suffix in ('.csv', '.md', '.png')},
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장:', OUT_DIR)
print((OUT_DIR / 'gain-summary.md').read_text(encoding='utf-8'))